# Reading subrun metadata from the experiment database

Pull run/subrun timing, ctag counts, and data-quality flags from the g-2 online database and assemble them into a single pandas dataframe via inner joins. (Connection details redacted.)

In [ ]:
import pandas as pd
import argparse,csv,datetime,time
import numpy as np
import matplotlib.mlab as mlab
import matplotlib.pyplot as plt
import math
import matplotlib.dates as mdate
import psycopg2
from collections import defaultdict
import sys

### First get the column names for the three tables we're interested in

* gm2dq.ctagswithdqc
* gm2dq.subrun_time
* gm2dq.subrun_status

__We also may want to compare with the table:__

* gm2ctag_dqm

In [ ]:
def connect(db):
    #connect to the database
    if (db ==  'localhost'):
        dsn  = "dbname=gm2_db user=db_user host=localhost port=5434"
    elif (db == 'db-host'):
        dsn  = "dbname=gm2_db user=db_user host=db-host port=5433"
    else:
        print ("None supported DB specified - run with --db=localhost or --db=db-host")
        sys.exit(-1)
    conn = psycopg2.connect(dsn)
    curr = conn.cursor()
    return conn,curr

In [ ]:
def get_labels(db='localhost', table='') :
    #connect
    conn,curr = connect(db)

    #get the column names for the specified table
    sql = "SELECT * FROM " + table +  " LIMIT 0"
    curr.execute(sql)
    colnames = [desc[0] for desc in curr.description]
    
    #close the connection without commiting any accidential changes
    conn.rollback()
    
    return colnames

In [ ]:
print(get_labels(table='gm2dq.ctagswithdqc'), '\n \n')
print(get_labels(table='gm2dq.subrun_time'), '\n \n')
print(get_labels(table='gm2dq.subrun_status'), '\n \n')
print(get_labels(table='gm2ctag_dqm'))

###  Get a list of runs/subruns for a subset of the 60 hour dataset

In [ ]:
def get_runs(start, end, db='localhost', table='gm2dq.subrun_time'):
    #connect
    conn,curr = connect(db)
 
    #Prepare the sql command and execute
    what = f"select run, subrun, start_time, end_time from {table} "
    where = f"where start_time >= '{start}' and end_time <= '{end}' "
    order = "order by start_time ASC"
    sql = what + where + order
    curr.execute(sql)
    
    #fetch the data
    rows = curr.fetchall()
    
    #Close the connection without commiting any accidential changes
    conn.rollback()
    
    return rows

In [ ]:
runs = get_runs(start="2018-04-22 00:00:00",end="2018-04-25 00:00:00")
df_time = pd.DataFrame.from_records(runs,columns=["run", "subrun", "start_time", "end_time"])
df_time.set_index(['run','subrun'],inplace=True)
df_time.head()

###  get the ctags for all the runs/subruns, 

In [ ]:
def get_ctags(db='localhost', table='gm2dq.ctagswithdqc'):
    #connect
    conn,curr = connect(db)
 
    #Prepare the sql command and execute
    what = f"select run, subrun, ctags from {table} "
    order = "order by run,subrun ASC"
    sql = what + order
    curr.execute(sql)
    
    #fetch the data
    rows = curr.fetchall()
    
    #Close the connection without commiting any accidential changes
    conn.rollback()
    
    return rows

In [ ]:
ctags = get_ctags()
df_ctag = pd.DataFrame.from_records(ctags,columns=["run", "subrun", "ctags"])
df_ctag.set_index(['run','subrun'],inplace=True)
df_ctag.head()

In [ ]:
def get_flags(db='localhost', table='gm2dq.subrun_status'):
    #connect
    conn,curr = connect(db)
 
    #Prepare the sql command and execute
    what = f"select * from {table} "
    order = "order by run,subrun ASC"
    sql = what + order
    curr.execute(sql)
    
    #fetch the data
    rows = curr.fetchall()
    
    #Close the connection without commiting any accidential changes
    conn.rollback()
    
    return rows

In [ ]:
flags = get_flags()
cols = get_labels(table='gm2dq.subrun_status')
df_flags = pd.DataFrame.from_records(flags,columns=cols)
df_flags.set_index(['run','subrun'],inplace=True)
df_flags.head()

### Do an inner join to get a df with all relevant information from the database

In [ ]:
df_total = pd.merge(df_time, df_ctag, on=['run','subrun'], how='inner')
df_total = pd.merge(df_total, df_flags, on=['run','subrun'], how='inner')
df_total.head()

### Get the poorly calibrated ctags for the entire 60 hour data

we may want them for later systematic studies

In [ ]:
def get_poor_ctags(start,end, db='localhost', table='gm2ctag_dqm') :

    #connect
    conn,curr = connect(db)
 
    #Prepare the sql command and execute
    what = f"select time, ctags from {table} "
    where = f"where time >= '{start}' and time <= '{end}' "
    order = "order by time ASC"
    sql = what + where + order
    curr.execute(sql)
    conn.commit()
    
    #fetch the data
    rows = curr.fetchall()

    return rows

In [ ]:
poor = get_poor_ctags(start="2018-04-22 00:00:00",end="2018-04-25 00:00:00")
df_poor = pd.DataFrame.from_records(poor,columns=["poor_time", "poor_ctags"])
df_poor.head()

### Write a function that returns a run/subrun based off a time

and apply that function to the dataframe of "poor" ctags

In [ ]:
def time_to_run(time,df):
    
    #Convert to date/time 
    time = pd.to_datetime(time)
    
    #select only the rows that span the specified time
    mask =  (time >  df['start_time']) & (time <=  df['end_time'])

    #Get the list of tuples of the runs/subruns
    ans = df[mask].index.tolist()
    
    #if the list is empty, set the run and subrun to -1
    if (ans == []):
        ans = [(-1,-1)]
    
    return ans

In [ ]:
#Just a test
time_to_run('2018-04-22 15:14:55',df_total)

In [ ]:
times = df_poor['poor_time'].tolist()
l = [time_to_run(t, df_total) for t in times]

In [ ]:
df_poor['run'] = [v[0][0] for v in l]
df_poor['subrun'] = [v[0][1] for v in l]
df_poor.set_index(['run','subrun'],inplace=True)
df_poor.tail()

### Do another inner join

In [ ]:
df_total = pd.merge(df_total, df_poor, on=['run','subrun'], how='inner')
df_total.head()

### Show the relationship between ctags and poor_ctags

In [ ]:
fig = plt.figure(figsize=(12,8))

mask = (df_total['ctags'] > 15000) & (df_total['poor_ctags'] > 400)
mask = df_total['ctags'] > -100
# mask = (df_total['ctags_ok'] == True) & (df_total['ctags_loose_ok'] == True) & (df_total['ctags_repeat_ok'] == True)
mask = (df_total['ctags_ok'] == True) & (df_total['field_ok'] == True)  & (df_total['ctags_loose_ok'] == True)


x = df_total[mask]['ctags']
y = df_total[mask]['poor_ctags']

plt.scatter(x, y, alpha=0.15,marker="o",s=150)

plt.xlabel("gm2dq.ctagswithdqc",fontsize=18)
plt.ylabel("gm2ctag_dqm",fontsize=18)

plt.grid()
plt.show()

In [ ]:
fig = plt.figure(figsize=(12,8))
df_total.plot('start_time','ctags',figsize=(10,4))
df_total.plot('start_time','poor_ctags',figsize=(10,4))


plt.xlabel("gm2dq.ctagswithdqc",fontsize=18)
plt.ylabel("gm2ctag_dqm",fontsize=18)

plt.grid()
plt.show()